# Read In, Clean, and Merge Data

## Read in the data

### Read in the gated station entry data

In [ ]:
# ALC
# Imports

import pandas as pd 
from os import walk
import re
import seaborn as sns
import matplotlib.pyplot as plt
import datetime
import math

In [ ]:
# ALC
# Gated Station Entries
# From the MBTA Open Data Portal
# https://mbta-massdot.opendata.arcgis.com/datasets/001c177f07594e7c99f193dde32284c9_0/explore
input_file_2022 = "../data/gated station entries/GSE_2022.csv"
input_file_2023 = "../data/gated station entries/GSE_2023.csv"
input_file_2024 = "../data/gated station entries/GSE_2024.csv"
input_file_2025 = "../data/gated station entries/GSE_2025.csv"

gse_2022 = pd.read_csv(input_file_2022)
gse_2023 = pd.read_csv(input_file_2023)
gse_2024 = pd.read_csv(input_file_2024)
gse_2025 = pd.read_csv(input_file_2025)

station_entries = pd.concat([gse_2022, gse_2023, gse_2024, gse_2025])

print(station_entries.shape)

### Read in the service alerts data

In [ ]:
# ALC
# these are as downloaded from the MBTA's Open Data Portal. There is one file
# for every month of the year. 
# https://mbta-massdot.opendata.arcgis.com/datasets/90ed9092bd7a4285b296d5ff938edf29_0/explore
# https://mbta-massdot.opendata.arcgis.com/datasets/f960d5089e164fb8b6c6936c70719d72/about
# https://mbta-massdot.opendata.arcgis.com/datasets/ef115f7cf6684360b040b6d1d033eff0/about
input_filenames = [[2022, ['alerts_2022_01', 'alerts_2022_02', 'alerts_2022_03', 
                           'alerts_2022_04', 'alerts_2022_05', 'alerts_2022_06', 
                           'alerts_2022_07', 'alerts_2022_08', 'alerts_2022_09', 
                           'alerts_2022_10', 'alerts_2022_11', 'alerts_2022_12']],
                   [2023, ['alerts_2023_01', 'alerts_2023_02', 'alerts_2023_03', 
                           'alerts_2023_04', 'alerts_2023_05', 'alerts_2023_06', 
                           'alerts_2023_07', 'alerts_2023_08', 'alerts_2023_09', 
                           'alerts_2023_10', 'alerts_2023_11', 'alerts_2023_12']],
                   [2024, ['2024-01_ALERTS', '2024-02_ALERTS', '2024-03_ALERTS', 
                           '2024-04_ALERTS', '2024-05_ALERTS', '2024-06_ALERTS', 
                           '2024-07_ALERTS', '2024-08_ALERTS', '2024-09_ALERTS', 
                           '2024-10_ALERTS','2024-11_ALERTS', '2024-12_ALERTS']],
                   [2025, ['2025-01_ALERTS', '2025-02_ALERTS', '2025-03_ALERTS', 
                           '2025-04_ALERTS', '2025-05_ALERTS', '2025-06_ALERTS']]]
# initialize array
alerts = []

for year, filenames in input_filenames:
    for filename in filenames:
        path = f'../data/service alerts/{year}/{filename}.csv'
        # including low_memory = false because we have some nas in columns
        month_alerts = pd.read_csv(path, quotechar='"', low_memory = False)
        alerts.append(month_alerts[month_alerts['effect_detail'] == 'DELAY'])
        
# put it all together
# this is HUGE when you keep every service alert type
service_alerts = pd.concat(alerts)
print(service_alerts.shape)

### Read in the speed restrictions data

In [ ]:
#ALC
# Speed Restriction Data from MBTA Open Data Portal
# https://mbta-massdot.opendata.arcgis.com/datasets/d73ed67e4cc84a84b818ea2c5caef696/about
slow_zone_files = []
# found this after doing service alerts above
# will focus more on switching to use the api than making consistent for now
for (root, dirs, filenames) in walk("../data/slow zones/"):
    slow_zone_files.extend(filenames)
    break
    
slow_zone_data = []

for file in slow_zone_files:
    path = f"../data/slow zones/{file}"
    if (re.search("\.csv$", file)):
        sz = pd.read_csv(path, quotechar='"')
        slow_zone_data.append(sz)
        
slow_zones = pd.concat(slow_zone_data)
slow_zones.shape

### Read in the reliability data

In [ ]:
# ALC
# MBTA Reliability data from the MBTA Open Data Portal
# https://mbta-massdot.opendata.arcgis.com/datasets/b3a24561c2104422a78b593e92b566d5_0/explore
reliability_input = "../data/reliability/MBTA Bus, Commuter Rail, & Rapid Transit Reliability.csv"
reliability = pd.read_csv(reliability_input)

reliability.shape

## Clean the data and prepare for join

### Make date format consistent

In [ ]:
#ALC
def get_date(date):
    return pd.to_datetime(date, format='mixed').dt.date

# the date in the reliability data appears yyyy/mm/dd hh:mm:ss+00
# we will use yyyy-mm-dd for service date
reliability['service_date'] = get_date(reliability['service_date'])

# slow zones uses what we expect but we just rename the column for consistency
slow_zones['Calendar_Date'] = get_date(slow_zones['Calendar_Date'])
slow_zones = slow_zones.rename(columns = {'Calendar_Date': 'service_date'})

# service_alerts is more complicated. there appear to be some problem
# rows. for example in the june 2025 dataset there is a detour alert from july
# of 2024 that never closed
service_alerts['active_period_start_date'] = get_date(service_alerts['active_period_start_date'])
service_alerts['active_period_start_dt'] = get_date(service_alerts['active_period_start_dt'])
service_alerts['active_period_end_dt'] = get_date(service_alerts['active_period_end_dt'])
service_alerts['created_dt'] = get_date(service_alerts['created_dt'])
service_alerts['closed_dt'] = get_date(service_alerts['closed_dt'])
station_entries['service_date'] = get_date(station_entries['service_date'])

### Make line consistent

In [ ]:
# ALC
def simplify_route(line):
    if pd.isna(line) or 'Silver Line' in line:
        return None
    elif 'Green Line' in line or 'Green-' in line:
        return 'green'
    elif 'Red Line' in line or 'Red' in line:
        return "red"
    elif 'Blue Line' in line or 'Blue' in line:
        return 'blue'
    elif 'Orange Line' in line or 'Orange' in line:
        return 'orange'
    else:
        return line

# station entry lines
station_entries['route_or_line'] = station_entries['route_or_line'].apply(simplify_route)

# there is silver line data in here which is out of scope
station_entries = station_entries[station_entries['route_or_line'].isin(['green', 'red', 'blue', 'orange'])]
station_entries = station_entries.rename(columns = {'gated_entries': 'est_ridership',
                                                   'route_or_line': 'line'})

# slow zones lines
slow_zones['line'] = slow_zones['Line'].apply(simplify_route)

# reliability lines, cr and rt
reliability['line'] = reliability['gtfs_route_long_name'].apply(simplify_route)

# this removes out of scope bus data
reliability = reliability[reliability['line'].notna()]

service_alerts['route_id'] = service_alerts['route_id'].apply(simplify_route)
service_alerts = service_alerts.rename(columns = {'route_id': 'line'})

### Aggregation

In [ ]:
# function to aggregate reliability data by date and line
# get one row per line per date with an aggregate score
def agg_reliability_data_daily(reliability):
    return reliability.groupby(['service_date', 'line'], as_index=False)[
        ['otp_numerator', 'otp_denominator']
    ].sum()
    
# gated station entries are given by the half hour. we will need to
# aggregate these down to the day
def agg_station_entries_daily(station_entries):
    return station_entries.groupby(['service_date', 'line'], as_index=False)[
               'est_ridership'
           ].sum()

# count and distane of slow zones daily
def agg_slow_zones_daily(slow_zones):
    return slow_zones.groupby(['service_date', 'line'], as_index=False).agg(
        num_slow_zones = ('ID', 'nunique'),
        total_track_pct = ('Line_Restricted_Track_Pct', 'sum'),
        total_miles_affected = ('Restriction_Distance_Miles', 'sum')
    )

## Relate the data

In [ ]:
# join datasets
# does it have to be done two at a time?
station_entries_reliability = pd.merge(
    agg_station_entries_daily(station_entries), 
    agg_reliability_data_daily(reliability), 
    on = ['service_date', 'line'], 
    how = 'inner')
    
# doing a left join here. if there is no data in the 
# slow zones file, it means slow zones = none
merged_data = pd.merge(
    station_entries_reliability,
    agg_slow_zones_daily(slow_zones),
    on = ['service_date', 'line'],
    how = 'left'
)

merged_data = merged_data[merged_data['service_date'] > datetime.date(2022, 12, 31)]

In [ ]:
merged_data['otp_score'] = merged_data['otp_numerator'] / merged_data['otp_denominator']
merged_data['num_slow_zones']

# if data is not present it means there were no slow zones
merged_data['num_slow_zones'] = merged_data['num_slow_zones'].fillna(0)
merged_data['total_track_pct'] = merged_data['total_track_pct'].fillna(0)
merged_data['total_miles_affected'] = merged_data['total_miles_affected'].fillna(0)
# these may be useful
merged_data['day_of_week'] = pd.to_datetime(merged_data['service_date']).dt.day_name()
merged_data['month'] = pd.to_datetime(merged_data['service_date']).dt.month_name()
# ideas to add: mark federal holidays, special event days, periods of known
# entire line closures, etc

merged_data

In [ ]:
merged_data.corr(numeric_only = True)

In [ ]:
# get some summary statistics for our numeric columns
pd.set_option('display.max_columns', None)
for x in ['est_ridership', 'otp_numerator', 'otp_denominator', 
        'num_slow_zones', 'total_track_pct', 'total_miles_affected', 
        'otp_score']:
    print(x)
    print(merged_data[x].describe())
    print(merged_data[x].skew())
    print(merged_data[x].kurt())

In [ ]:
# get some proportions and frequencies for our categorical variables
for line in merged_data['line'].unique():
    print(line + " appears " + str(len(merged_data[merged_data['line'] == line])) 
          + " times")
    print(line + " appears " + str(len(merged_data[merged_data['line'] == line])/len(merged_data) * 100)
          + " % of the time")
    
    
for line in merged_data['day_of_week'].unique():
    print(line + " appears " + str(len(merged_data[merged_data['day_of_week'] == line])) 
          + " times")
    print(line + " appears " + str(len(merged_data[merged_data['day_of_week'] == line])/len(merged_data) * 100)
          + " % of the time")
    
for line in merged_data['month'].unique():
    print(line + " appears " + str(len(merged_data[merged_data['month'] == line])) 
          + " times")
    print(line + " appears " + str(len(merged_data[merged_data['month'] == line])/len(merged_data) * 100)
          + " % of the time")

In [ ]:
# 1st plot, show how day of the week may be something we need
# to handle
plt.figure(figsize = (10, 6))

# have colors match line
palette = {'green': 'green', 'blue': 'blue', 'orange': 'orange', 'red': 'red'}

sns.barplot(
    data = merged_data,
    x = 'day_of_week',
    y = 'est_ridership',
    hue = 'line',
    estimator = 'median',
    errorbar = None,
    palette = palette
)

# labels
plt.title('Median Gated Station Entries by Day of Week and Line')
plt.ylabel('Median Gated Station Entries')
plt.xlabel('Day of Week')
# make better labels
plt.legend(title = 'Line', 
           labels = ['Blue Line', 'Green Line', 'Orange Line', 'Red Line'])
plt.tight_layout()
plt.palette = palette
plt.show()

In [ ]:
# we'll need to look at train lines individually
plt.figure(figsize = (10, 6))

# have colors match line
palette = {'green': 'green', 'blue': 'blue', 'orange': 'orange', 'red': 'red'}

sns.boxplot(x = 'line', 
            y = 'est_ridership',
            hue = 'line',
            palette = palette,
            data = merged_data)

lines = ['Blue Line', 'Green Line', 'Orange Line', 'Red Line']
# labels
plt.title('Gated Station Entries per Rapid Transit Line')
plt.xlabel('Line')
plt.ylabel('Gated Station Entries')
plt.gca().yaxis.set_major_formatter(lambda x, pos: f'{x:,.0f}')
plt.xticks(ticks = [0, 1, 2, 3], labels = lines)
# make better labels
plt.legend()
plt.tight_layout()
plt.palette = palette
plt.show()

In [ ]:
plt.figure(figsize = (10, 6))

sns.lineplot(x = 'service_date', 
            y = 'est_ridership',
             color = 'green',
            data = merged_data[merged_data['line'] == 'green'])
# labels
plt.title('Green Line Gated Station Entries')
plt.xlabel('Date')
plt.ylabel('Gated Station Entries')
plt.gca().yaxis.set_major_formatter(lambda x, pos: f'{x:,.0f}')
plt.tight_layout()
plt.show()